# 6.4. 多输入多输出通道

彩色图像和中间特征图通常包含多个通道。多通道卷积需要对每个输入通道分别进行互相关，再求和；多个输出通道对应多个独立的卷积核组。

### 6.4.1 多输入通道

实现 `corr2d_multi_in(X, K)`：对每个输入通道计算二维互相关并求和。用形状为 `(2, 3, 3)` 的输入和对应卷积核验证结果。

> **易错提醒：按对应输入通道配对，再将互相关结果相加。**
>
> - `for x, k in zip(X, K)` 会依次配对 `(X[0], K[0])`、`(X[1], K[1])`，因此这里应使用 `zip`。输入和卷积核的通道数应相同；`zip` 会在较短的一方结束时停止，不会自动检查通道数是否一致。
> - `for x, k in X, K` 等价于遍历元组 `(X, K)`，然后将每个张量拆包给 `x` 和 `k`。本例中两次配对是 `(X[0], X[1])`、`(K[0], K[1])`，不是对应通道；通道数不为 2 时还会报拆包错误。
> - `for x in X` 内再写 `for k in K` 会计算所有通道组合。本例会累加 4 项，而正确计算只累加 `d2l.corr2d(X[0], K[0])` 和 `d2l.corr2d(X[1], K[1])` 这 2 项。
> - 在零张量上用 `res += d2l.corr2d(x, k)` 累加对应通道的结果，与 `return sum(d2l.corr2d(x, k) for x, k in zip(X, K))` 等价。本例输出形状为 `(2, 2)`；由于 `X` 随机生成，每次运行的具体数值通常不同。

In [5]:
import torch
from d2l import torch as d2l
def corr2d_multi_in(X,K):
    
    res=torch.zeros_like(d2l.corr2d(X[0],K[0]))
    for x,k in zip(X,K):
            res+=d2l.corr2d(x,k)
    return res
    ##//return sum(d2l.corr2d(x, k) for x, k in zip(X, K))
X=torch.rand(2,3,3)
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
print(corr2d_multi_in(X,K))


tensor([[7.4504, 5.3084],
        [7.9607, 6.6042]])


### 6.4.2 多输出通道

实现 `corr2d_multi_in_out(X, K)`：遍历输出通道，对每个三维卷积核调用多输入通道运算并堆叠结果。说明输出通道数如何增加模型能够学习的特征种类。

In [7]:
def corr2d_multi_in_out(X,K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
    
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

### 6.4.3 1×1 卷积层

1×1 卷积不混合空间位置，只在通道维度上做线性组合。实现 `corr2d_multi_in_out_1x1`，并验证它等价于对每个像素位置独立应用一个全连接层。



In [ ]:
def corr2d_multi_in_out_1x1(X,K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    K=K.reshape(c_)
    K = K.reshape((c_o, c_i))


### 6.4.4 小结与练习

练习：计算多输入多输出卷积的参数量；比较 1×1 卷积和全连接层的计算；尝试改变输入、输出通道数。